# Flow matching for simple labelled distributions

In this notebook we will show how flow matching and rectified flow can be used to sample data from a toy 2D distribution with labels.

In [ ]:
%load_ext autoreload
%autoreload 2

import data  # Auxiliary module for generating toy datasets
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Load toy labelled data

Generate target distribution data and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

target_data, target_labels = data.generate_toy_data("two_gaussians_supervised", n=1000)
unique_labels = set(target_labels)

num_couplings= 100000
source_data = np.random.randn(num_couplings, 2)

plotting.plot_distributions(source_data, target_data, target_labels=target_labels)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings, target_labels=target_labels)
plotting.plot_distributions(source_data, target_data, target_labels=target_labels, couplings=couplings)

Train a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

The trained network has the following structure, where the class label is encoded has an embedding vector:

In [ ]:
plotting.plot_network(network, couplings, save_filename="network_architecture")

Visualize the velocity fields we have learned: unconditional and conditional for both classes.

In [ ]:
plotting.plot_class_conditional_velocity_fields(network, source_data, target_data, target_labels=target_labels)

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow (for now with no labels)

In [ ]:
source_data = np.random.randn(1000, 2)
trajectories = models.compute_trajectories(network, source_data, labels=None)
plotting.animate_trajectories(trajectories, target_data=target_data)

We can also generate trajectories conditioned on the class labels to lead the generation process towards a particular class

In [ ]:
source_data = np.random.randn(1000, 2)
source_labels = np.random.choice(list(unique_labels), size=source_data.shape[0])
conditioned_trajectories = models.compute_trajectories(network, source_data, labels=source_labels)
plotting.animate_trajectories(conditioned_trajectories, target_data=target_data, labels=source_labels)

## Rectified flows

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=num_couplings, target_labels=target_labels)

num_reflow_steps = 2
for i in range(num_reflow_steps):
    print(f"Reflow step {i + 1}/{num_reflow_steps}...")
    couplings, rectified_network = models.reflow(couplings, model_arguments={"verbose": True})

In [ ]:
source_data = np.random.randn(1000, 2)
source_labels = np.random.choice(list(unique_labels), size=source_data.shape[0])
rectified_conditioned_trajectories = models.compute_trajectories(rectified_network, source_data, labels=source_labels)
plotting.animate_trajectories(rectified_conditioned_trajectories, target_data=target_data, labels=source_labels)

In [ ]:
plotting.plot_class_conditioned_generated_data_comparison(target_data, rectified_conditioned_trajectories, trajectories_labels=source_labels, target_labels=target_labels)